# Figure 2 Revisada: PD vs SNR — 1D CNN vs Classical (Xie 2021)

**Objective**: α-constrained comparison.  Both methods evaluated at the same FPR ≤ α = 10⁻⁷.

**Threshold setting** — Braca (2022) D3F Eq. 20:  
From H0 validation scores, fit Gaussian → τ*(α) = μ_H0 + Q⁻¹(1−α)·σ_H0  
(asymptotically valid; Q⁻¹(1−10⁻⁷) ≈ 5.199 standard deviations above H0 mean)

| Method | Test statistic | Threshold source |
|--------|---------------|-----------------|
| Classical (Xie 2021) | τ_eq = |Σ(y/h − ρ_s·msg)·tag_ref| / ρ_t | D3F Gaussian fit on τ_eq H0 val scores |
| 1D CNN (Chin & Chin 2025) | P(H1) ∈ [0,1] | D3F Gaussian fit on CNN H0 val scores |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import json
from scipy.stats import norm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from tensorflow import keras

print("TF:", keras.__version__)

# --- Paths ---
project_root      = Path.cwd().parent          # Redes Neurais/
results_dir       = project_root / "results"
data_dir          = results_dir / "data"
models_dir        = results_dir / "models"
visualizations_dir= results_dir / "visualizations"
visualizations_dir.mkdir(parents=True, exist_ok=True)

# --- Load dataset (from NN_01 v3) ---
dataset_path = data_dir / "dataset_cnn_yeq_0_30dB.h5"
if not dataset_path.exists():
    raise FileNotFoundError(f"Run NN_01_DataGeneration.ipynb first.\nExpected: {dataset_path}")

with h5py.File(str(dataset_path), 'r') as f:
    L_FIXED = int(f.attrs['L_FIXED'])

    Y_val   = f['val/y_eq'][:]          # (N_val, 1024) float32 — CNN input
    TAU_val = f['val/tau_eq'][:]        # (N_val,) float32 — classical scalar
    SNR_val = f['val/snr'][:]
    LBL_val = f['val/y'][:].astype(int)

    Y_test   = f['test/y_eq'][:]
    TAU_test = f['test/tau_eq'][:]
    SNR_test = f['test/snr'][:]
    LBL_test = f['test/y'][:].astype(int)

print(f"Val  : {Y_val.shape}  H1={LBL_val.sum()} H0={(1-LBL_val).sum()}")
print(f"Test : {Y_test.shape}  H1={LBL_test.sum()} H0={(1-LBL_test).sum()}")
print(f"SNR val  : [{SNR_val.min():.1f}, {SNR_val.max():.1f}] dB")
print(f"SNR test : [{SNR_test.min():.1f}, {SNR_test.max():.1f}] dB")

# --- Load trained CNN ---
model_path = models_dir / 'cnn1d_tag_auth_best.keras'
if not model_path.exists():
    raise FileNotFoundError(f"Train the CNN first in NN_02_DNN_Correlator.ipynb.\nExpected: {model_path}")

cnn = keras.models.load_model(str(model_path))
print(f"\nCNN model loaded: {model_path.name}")
print(f"Input shape: {cnn.input_shape}")

## Step 1 — Compute CNN scores on validation set H0 samples

Run CNN inference on the val set to get P(H1) scores for every sample.
These H0 scores are used to calibrate the D3F threshold per SNR bin.

In [ ]:
# CNN inference on full val set (batch to avoid OOM)
P_val = cnn.predict(Y_val.reshape(-1, L_FIXED, 1), batch_size=512, verbose=1).flatten()
print(f"P_val shape: {P_val.shape}")
print(f"  H0 scores: mean={P_val[LBL_val==0].mean():.4f}, std={P_val[LBL_val==0].std():.4f}")
print(f"  H1 scores: mean={P_val[LBL_val==1].mean():.4f}, std={P_val[LBL_val==1].std():.4f}")

# CNN inference on test set
P_test = cnn.predict(Y_test.reshape(-1, L_FIXED, 1), batch_size=512, verbose=0).flatten()
print(f"\nP_test shape: {P_test.shape}")

## Step 2 — D3F Threshold Calibration per SNR bin (Braca 2022, Eq. 20)

For each SNR bin, collect H0 validation scores, fit a Gaussian, and extrapolate:

**τ*(α) = μ_H0 + Q⁻¹(1−α) · σ_H0**

where Q⁻¹(1−10⁻⁷) = norm.ppf(1 − 10⁻⁷) ≈ 5.199

This is applied to both:
- **CNN** scores: P(H1) ∈ [0,1]
- **Classical** scores: τ_eq ∈ [0,∞)

In [ ]:
# SNR bins for Figure 2 (0,5,10,...,30 dB)
SNR_POINTS = np.arange(0, 31, 5)     # bin centres
HALF_BIN   = 2.5                       # ± dB around each centre
ALPHA      = 1e-7
q_inv      = norm.ppf(1 - ALPHA)      # ≈ 5.199
print(f"Q⁻¹(1 − α) = Q⁻¹(1 − 10⁻⁷) = {q_inv:.4f}")

thresholds_cnn  = {}   # SNR → τ*_cnn
thresholds_tau  = {}   # SNR → τ*_classical

print(f"\n{'SNR':>5}  {'N_H0_val':>9}  {'μ_H0_cnn':>10}  {'σ_H0_cnn':>10}  {'τ*_cnn':>10}  {'τ*_tau':>12}")
print("-" * 72)

for snr in SNR_POINTS:
    mask_h0 = (LBL_val == 0) & (SNR_val >= snr - HALF_BIN) & (SNR_val < snr + HALF_BIN)
    n_h0 = mask_h0.sum()

    if n_h0 < 30:
        thresholds_cnn[snr] = np.nan
        thresholds_tau[snr] = np.nan
        print(f"{snr:>5}  {n_h0:>9}  {'N/A':>10}")
        continue

    # CNN scores under H0 at this SNR
    s_cnn = P_val[mask_h0]
    mu_c, sig_c = s_cnn.mean(), s_cnn.std()
    tau_star_cnn = mu_c + q_inv * sig_c
    tau_star_cnn = min(tau_star_cnn, 1.0)   # clip to [0,1] (P(H1) range)

    # Classical tau_eq scores under H0 at this SNR
    s_tau = TAU_val[mask_h0]
    mu_t, sig_t = s_tau.mean(), s_tau.std()
    tau_star_tau = mu_t + q_inv * sig_t

    thresholds_cnn[snr] = float(tau_star_cnn)
    thresholds_tau[snr] = float(tau_star_tau)

    print(f"{snr:>5}  {n_h0:>9}  {mu_c:>10.5f}  {sig_c:>10.5f}  {tau_star_cnn:>10.6f}  {tau_star_tau:>12.4f}")

## Step 3 — Compute PD vs SNR for both methods

In [ ]:
pd_cnn  = {}
pd_tau  = {}
n_h1_per_bin = {}

print(f"\n{'SNR':>5}  {'N_H1_test':>10}  {'PD_CNN':>10}  {'PD_Classical':>14}")
print("-" * 46)

for snr in SNR_POINTS:
    mask_h1 = (LBL_test == 1) & (SNR_test >= snr - HALF_BIN) & (SNR_test < snr + HALF_BIN)
    n_h1 = mask_h1.sum()
    n_h1_per_bin[snr] = int(n_h1)

    if n_h1 < 5 or np.isnan(thresholds_cnn.get(snr, np.nan)):
        pd_cnn[snr] = np.nan
        pd_tau[snr] = np.nan
        print(f"{snr:>5}  {n_h1:>10}  {'N/A':>10}")
        continue

    p_h1  = P_test[mask_h1]
    tau_h1= TAU_test[mask_h1]

    pd_c = float((p_h1  >= thresholds_cnn[snr]).mean())
    pd_t = float((tau_h1 >= thresholds_tau[snr]).mean())

    pd_cnn[snr] = pd_c
    pd_tau[snr] = pd_t

    print(f"{snr:>5}  {n_h1:>10}  {pd_c:>10.4f}  {pd_t:>14.4f}")

## Figure 2 — PD vs SNR, α-constrained

In [ ]:
snr_arr  = np.array(SNR_POINTS, dtype=float)
pd_c_arr = np.array([pd_cnn.get(s, np.nan) for s in SNR_POINTS])
pd_t_arr = np.array([pd_tau.get(s, np.nan) for s in SNR_POINTS])

fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(snr_arr, pd_t_arr, 'o-',  color='tomato',    linewidth=2.5, markersize=8,
        markerfacecolor='white', markeredgewidth=2, label='Classical Auth-SUP (Xie 2021)')
ax.plot(snr_arr, pd_c_arr, 's--', color='steelblue', linewidth=2.5, markersize=8,
        markerfacecolor='white', markeredgewidth=2, label='1D CNN (Chin & Chin 2025)')

ax.set_xlabel('SNR (dB)', fontsize=13)
ax.set_ylabel('Probability of Detection (PD)', fontsize=13)
ax.set_title(f'Figure 2 (Revised) — PD vs SNR\nBoth methods at FPR ≤ α = 10⁻⁷ (D3F threshold, Braca 2022)',
             fontsize=13)
ax.set_xlim(-1, 31)
ax.set_ylim(-0.05, 1.05)
ax.set_xticks(SNR_POINTS)
ax.set_yticks(np.arange(0, 1.1, 0.1))
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)

# Annotation: system params
info = (f"System: BPSK | Rayleigh | L=1024\n"
        f"α = FPR ≤ {ALPHA:.0e}\n"
        f"Threshold via D3F Gaussian (Braca 2022)")
ax.text(0.02, 0.98, info, transform=ax.transAxes, fontsize=9,
        va='top', ha='left',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
fig_path = visualizations_dir / "Figure2_PD_vs_SNR_alpha_constrained.png"
plt.savefig(str(fig_path), dpi=150)
plt.show()
print(f"Saved → {fig_path}")

In [ ]:
# Save numerical results for the paper
results = {
    'alpha': ALPHA,
    'method_cnn':  {'label': '1D CNN (Chin & Chin 2025)', 'PD_vs_SNR': pd_cnn},
    'method_tau':  {'label': 'Classical Auth-SUP (Xie 2021)', 'PD_vs_SNR': pd_tau},
    'thresholds':  {'cnn': thresholds_cnn, 'classical': thresholds_tau},
}

out_path = results_dir / "data" / "figure2_pd_vs_snr.json"
with open(str(out_path), 'w') as f:
    # Convert int keys to str for JSON
    def to_str_keys(d):
        return {str(k): v for k, v in d.items()}
    results['method_cnn']['PD_vs_SNR']  = to_str_keys(pd_cnn)
    results['method_tau']['PD_vs_SNR']  = to_str_keys(pd_tau)
    results['thresholds']['cnn']        = to_str_keys(thresholds_cnn)
    results['thresholds']['classical']  = to_str_keys(thresholds_tau)
    json.dump(results, f, indent=2)

print(f"Saved numerical results → {out_path}")